# 05 — Évaluation finale sur le test (une seule fois)

> ⚠️ **C'est le seul endroit du projet où `test.csv` est ouvert.**
> Le modèle (arbre de décision), ses variables et la règle de référence ont été choisis dans `04_modeles`, sur le train uniquement. **Rien n'est modifié d'après les résultats ci-dessous.**

Le modèle retenu est réentraîné sur **tout** le train, puis appliqué une seule fois au test. La règle simple est évaluée sur le même test, pour comparaison.

In [1]:
import sys
from pathlib import Path

# Racine du repo = dossier qui contient config.py
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(RACINE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import modele_utils as mu

pd.set_option("display.float_format", "{:,.3f}".format)
FIGURES = RACINE / "figures"
RESULTATS = RACINE / "resultats"
FIGURES.mkdir(exist_ok=True)
RESULTATS.mkdir(exist_ok=True)
print("Graine :", config.SEED, "| Variables :", config.FEATURES_MODELE)

Graine : 42 | Variables : ['Customer Type', 'Type of Travel', 'Class', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']


In [2]:
df_train, y_train = mu.charger_train()
modele = mu.MODELES[mu.MODELE_RETENU]().fit(df_train[config.FEATURES_MODELE], y_train)
print("Modèle :", mu.MODELE_RETENU, "| Règle de référence :", mu.REGLE_DE_REFERENCE)

Modèle : Arbre de décision | Règle de référence : Voyage personnel


## Lecture unique du test

In [3]:
df_test, y_test = mu.charger_test()   # <- UNIQUE lecture du fichier test
print(f"{len(df_test):,} passagers dans le test, dont {y_test.mean():.1%} « neutral or dissatisfied »")

score_modele = modele.predict_proba(df_test[config.FEATURES_MODELE])[:, 1]
score_regle = mu.REGLES[mu.REGLE_DE_REFERENCE](df_test).to_numpy()

25,976 passagers dans le test, dont 56.1% « neutral or dissatisfied »


## Résultats

In [4]:
final = pd.DataFrame({
    f"Règle : {mu.REGLE_DE_REFERENCE}": mu.evaluer(y_test, score_regle),
    mu.MODELE_RETENU: mu.evaluer(y_test, score_modele),
}).T
final.loc["Modèle parfait (borne)"] = {
    "Précision top 10 %": 1.0, "Gain / 1000 passagers (€)": mu.gain_parfait_par_1000(y_test)}
final["Gain vs règle simple (€ / 1000)"] = (
    final["Gain / 1000 passagers (€)"] - final["Gain / 1000 passagers (€)"].iloc[0])
effectifs = ["Passagers abordés", "Bons choix", "Mauvais choix", "Gain total (€)"]
final[effectifs] = final[effectifs].round(0)
final.to_csv(RESULTATS / "05_resultats_test.csv")
final

,AUC,Précision top 10 %,Gain / 1000 passagers (€),Passagers abordés,Bons choix,Mauvais choix,Gain total (€),Gain vs règle simple (€ / 1000)
Règle : Voyage personnel,0.710,0.900,"5,001.008","2,598.000","2,338.000",259.000,"129,906.000",0.000
Arbre de décision,0.845,0.961,"5,606.616","2,598.000","2,495.000",102.000,"145,637.000",605.608
Modèle parfait (borne),NaN,1.000,"6,000.000",NaN,NaN,NaN,NaN,998.992


## Les erreurs du modèle, en clair

In [5]:
r = final.loc[mu.MODELE_RETENU]
insatisfaits = y_test.sum()
print(f"Passagers dans le test                     : {len(y_test):,}")
print(f"Passagers abordés (10 %)                   : {r['Passagers abordés']:,.0f}")
print(f"  vraiment mécontents   (+{mu.GAIN_BON_CHOIX:.0f} € chacun)  : {r['Bons choix']:,.0f}")
print(f"  en fait satisfaits    ({mu.GAIN_MAUVAIS_CHOIX:.0f} € chacun)  : {r['Mauvais choix']:,.0f}")
print(f"Mécontents non abordés (capacité limitée)  : {insatisfaits - r['Bons choix']:,.0f}")
print(f"Gain total sur le test                     : {r['Gain total (€)']:,.0f} €")
print(f"Gain pour 1000 passagers                   : {r['Gain / 1000 passagers (€)']:,.0f} €")

Passagers dans le test                     : 25,976
Passagers abordés (10 %)                   : 2,598
  vraiment mécontents   (+60 € chacun)  : 2,495
  en fait satisfaits    (-40 € chacun)  : 102
Mécontents non abordés (capacité limitée)  : 12,078
Gain total sur le test                     : 145,637 €
Gain pour 1000 passagers                   : 5,607 €


**Limite :** le test, comme le train, ne contient que des **répondants** à l'enquête. Le modèle sera pourtant appliqué à **tous** les passagers. Si les non-répondants diffèrent, la précision réelle peut être plus faible. Il faudra un pilote sur quelques vols avant de généraliser.